# Phase 1 — Data Acquisition & Preparation
**Project:** Sentiment Analysis of Yelp Reviews at Scale (Apache Spark + NLP)

**Goal of this notebook:** download the Yelp Open Dataset, load it with Spark, clean it, create sentiment labels, and save a clean Parquet copy to Google Drive so the next phases start from the same data.

| Stars | Label | Meaning |
|---|---|---|
| 4 – 5 | 1 | Positive |
| 1 – 2 | 0 | Negative |
| 3 | removed | Neutral |

**Outputs (on Google Drive):**
- `yelp_project/reviews_clean.parquet`
- `yelp_project/business.parquet`

> Run on Google Colab. Use a **High-RAM** runtime if available.
> ⚠️ Never put your Kaggle token in the notebook or commit it to GitHub.

## 0. Configuration
Change `BASE_PATH` if the shared project folder is in a different location on your Drive.

In [1]:
BASE_PATH = "/content/drive/MyDrive/yelp_project"   # shared project folder on Drive
RAW_PATH  = "/content/yelp"                          # local (temporary) download folder in Colab

REVIEWS_OUT  = f"{BASE_PATH}/reviews_clean.parquet"
BUSINESS_OUT = f"{BASE_PATH}/business.parquet"

## 1. Environment setup

In [3]:
!pip install -q pyspark

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(BASE_PATH, exist_ok=True)

Mounted at /content/drive


## 2. Download the dataset from Kaggle
1. Get your Kaggle API token from https://www.kaggle.com/settings (API section).
2. Run the cell below and paste the token when asked. It is hidden and is **not** saved in the notebook.

> Skip sections 2–7 if the Parquet files already exist on Drive.

In [4]:
!pip install -q -U kaggle

import os
from getpass import getpass

# paste your Kaggle API token when asked (never hard-code it in the notebook)
os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token: ")

Paste your Kaggle API token: ··········


In [5]:
!kaggle datasets download -d yelp-dataset/yelp-dataset -p {RAW_PATH}
!unzip -q -o {RAW_PATH}/yelp-dataset.zip -d {RAW_PATH}
!rm {RAW_PATH}/yelp-dataset.zip
!ls -lh {RAW_PATH}

Dataset URL: https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset
License(s): other
100% 4.07G/4.07G [00:33<00:00, 132MB/s]

total 8.7G
-rw-r--r-- 1 root root  79K Mar 17  2022 Dataset_User_Agreement.pdf
-rw-r--r-- 1 root root 114M Mar 17  2022 yelp_academic_dataset_business.json
-rw-r--r-- 1 root root 274M Mar 17  2022 yelp_academic_dataset_checkin.json
-rw-r--r-- 1 root root 5.0G Mar 17  2022 yelp_academic_dataset_review.json
-rw-r--r-- 1 root root 173M Mar 17  2022 yelp_academic_dataset_tip.json
-rw-r--r-- 1 root root 3.2G Mar 17  2022 yelp_academic_dataset_user.json


## 3. Start Spark

In [6]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("YelpSentiment-Phase1")
         .config("spark.driver.memory", "10g")
         .config("spark.sql.shuffle.partitions", "64")
         .getOrCreate())

spark

## 4. Load the reviews
We pass an **explicit schema** so Spark does not scan the whole 5 GB file just to infer the column types (much faster).

In [7]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

review_schema = StructType([
    StructField("review_id",   StringType()),
    StructField("user_id",     StringType()),
    StructField("business_id", StringType()),
    StructField("stars",       DoubleType()),
    StructField("useful",      LongType()),
    StructField("funny",       LongType()),
    StructField("cool",        LongType()),
    StructField("text",        StringType()),
    StructField("date",        StringType()),
])

reviews = spark.read.schema(review_schema).json(f"{RAW_PATH}/yelp_academic_dataset_review.json")
reviews.printSchema()

total_reviews = reviews.count()
print(f"Total raw reviews: {total_reviews:,}")
reviews.select("stars", "text", "date").show(5, truncate=80)

root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- useful: long (nullable = true)
 |-- funny: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- text: string (nullable = true)
 |-- date: string (nullable = true)

Total raw reviews: 6,990,280
+-----+--------------------------------------------------------------------------------+-------------------+
|stars|                                                                            text|               date|
+-----+--------------------------------------------------------------------------------+-------------------+
|  3.0|If you decide to eat here, just be aware it is going to take about 2 hours fr...|2018-07-07 22:09:11|
|  5.0|I've taken a lot of spin classes over the years, and nothing compares to the ...|2012-01-03 15:28:18|
|  3.0|Family diner. Had the buffet. Eclectic assortment: a large chicken leg, frie

### 4.1 Original star distribution (before cleaning)

In [8]:
reviews.groupBy("stars").count().orderBy("stars").show()

+-----+-------+
|stars|  count|
+-----+-------+
|  1.0|1069561|
|  2.0| 544240|
|  3.0| 691934|
|  4.0|1452918|
|  5.0|3231627|
+-----+-------+



## 5. Clean the data and create labels
- drop rows with missing text or stars
- drop duplicate reviews
- drop empty / very short reviews (≤ 10 characters)
- drop neutral reviews (3 stars)
- create `label` (1 = positive, 0 = negative)
- keep `date` and `year` → needed for time-based analysis later

In [9]:
from pyspark.sql.functions import col, when, to_timestamp, to_date, year, length, trim

df = (reviews
      .select("review_id", "business_id", "stars", "text", "date")
      .dropna(subset=["text", "stars"])
      .dropDuplicates(["review_id"])
      .withColumn("text", trim(col("text")))
      .filter(length(col("text")) > 10)
      .filter(col("stars") != 3)
      .withColumn("label", when(col("stars") >= 4, 1).otherwise(0))
      .withColumn("date", to_date(to_timestamp(col("date"))))
      .withColumn("year", year(col("date"))))

df.cache()
clean_reviews = df.count()
print(f"Reviews after cleaning: {clean_reviews:,}")
print(f"Removed: {total_reviews - clean_reviews:,}")

Reviews after cleaning: 6,298,179
Removed: 692,101


### 5.1 Label distribution (class balance)

In [10]:
label_counts = {r["label"]: r["count"] for r in df.groupBy("label").count().collect()}
neg, pos = label_counts.get(0, 0), label_counts.get(1, 0)

print(f"Positive (1): {pos:>10,}  ({pos / clean_reviews:.1%})")
print(f"Negative (0): {neg:>10,}  ({neg / clean_reviews:.1%})")
print(f"Imbalance ratio (positive : negative) = {pos / neg:.2f} : 1")

Positive (1):  4,684,444  (74.4%)
Negative (0):  1,613,735  (25.6%)
Imbalance ratio (positive : negative) = 2.90 : 1


### 5.2 Reviews per year

In [11]:
df.groupBy("year").count().orderBy("year").show(30)

+----+------+
|year| count|
+----+------+
|2005|   683|
|2006|  3201|
|2007| 12458|
|2008| 39220|
|2009| 61614|
|2010|115549|
|2011|194809|
|2012|245187|
|2013|332536|
|2014|461656|
|2015|616159|
|2016|686637|
|2017|744886|
|2018|829238|
|2019|832821|
|2020|517544|
|2021|574499|
|2022| 29482|
+----+------+



## 6. Load the business data (used in Phase 4 for insights by city / category)

In [12]:
business = (spark.read.json(f"{RAW_PATH}/yelp_academic_dataset_business.json")
            .select("business_id", "name", "city", "state", "categories"))

print(f"Businesses: {business.count():,}")
business.show(5, truncate=50)

Businesses: 150,346
+----------------------+------------------------+-------------+-----+--------------------------------------------------+
|           business_id|                    name|         city|state|                                        categories|
+----------------------+------------------------+-------------+-----+--------------------------------------------------+
|Pns2l4eNsfO8kk83dixA6A|Abby Rappoport, LAC, CMQ|Santa Barbara|   CA|Doctors, Traditional Chinese Medicine, Naturopa...|
|mpf3x-BjTdTEA3yCZrAYPw|           The UPS Store|       Affton|   MO|Shipping Centers, Local Services, Notaries, Mai...|
|tUFrWirKiKi_TAnsVWINQQ|                  Target|       Tucson|   AZ|Department Stores, Shopping, Fashion, Home & Ga...|
|MTSW4McQd7CbVtyjqoe9mw|      St Honore Pastries| Philadelphia|   PA|Restaurants, Food, Bubble Tea, Coffee & Tea, Ba...|
|mWMc6_wTdE0EUBKIGXDVfA|Perkiomen Valley Brewery|   Green Lane|   PA|                         Brewpubs, Breweries, Food|
+-----------

## 7. Save as Parquet to Google Drive
`repartition` keeps the number of output files reasonable, which makes writing to / reading from Drive faster.

In [13]:
df.repartition(32).write.mode("overwrite").parquet(REVIEWS_OUT)
business.repartition(4).write.mode("overwrite").parquet(BUSINESS_OUT)
print("Saved to:", BASE_PATH)

Saved to: /content/drive/MyDrive/yelp_project


## 8. Verify the saved data

In [14]:
check = spark.read.parquet(REVIEWS_OUT)
print(f"Rows in saved Parquet: {check.count():,}")
check.printSchema()
check.show(3, truncate=60)

Rows in saved Parquet: 6,298,179
root
 |-- review_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- date: date (nullable = true)
 |-- label: integer (nullable = true)
 |-- year: integer (nullable = true)

+----------------------+----------------------+-----+------------------------------------------------------------+----------+-----+----+
|             review_id|           business_id|stars|                                                        text|      date|label|year|
+----------------------+----------------------+-----+------------------------------------------------------------+----------+-----+----+
|k4DCN72laEzIMxOBjLxPGA|-T5SfUI_QqgBrkq4D8Jn7w|  4.0|Great and friendly service! I went here for brunch with m...|2019-04-07|    1|2019|
|fr4hZ2U7fVyOxd9J2UlSwQ|A_AEG20YPuVW7Ao7gkHicg|  4.0|Great place for a snack or a full blown lunch, serves bee...|2019-04-20|    1|2019|
|HxqLRmQle

## 9. Phase 1 summary

| Metric | Value |
|---|---|
| Total raw reviews | 6,990,280 |
| Removed (3-star neutral: 691,934 + short/duplicate: 167) | 692,101 |
| Reviews after cleaning | 6,298,179 |
| Positive (label 1) | 4,684,444 (74.4%) |
| Negative (label 0) | 1,613,735 (25.6%) |
| Imbalance ratio (positive : negative) | 2.90 : 1 |
| Year range | 2005 – 2022 (2022 is partial, ~29K reviews) |
| Businesses | 150,346 |

**Key observations for the next phases**
- The data is **imbalanced (≈ 3 : 1)** → use class weights and report F1 / Recall, not only Accuracy.
- Review volume grows every year until 2019, then **drops in 2020 (COVID-19)** and recovers in 2021.
- 2022 contains only a few weeks of data → treat it with care in any time-based analysis.

---
### How to start the next phases
Start every new notebook with:
```python
df       = spark.read.parquet("/content/drive/MyDrive/yelp_project/reviews_clean.parquet")
business = spark.read.parquet("/content/drive/MyDrive/yelp_project/business.parquet")

# while developing: work on a 5% sample, then run the final version on all data
sample = df.sample(fraction=0.05, seed=42)
```